In [2]:

import os
import sys
from pathlib import Path
from typing import List, Dict, Optional

# Import the dynamic trainer
from abbvisionsystem.training_pipeline.yolo_trainer import (
    YOLO11DefectDetector, 
    evaluate_yolo11_model
)
from abbvisionsystem.training_pipeline.data_manager import (
    organize_dataset, 
    prepare_yolo_dataset, 
    augment_with_backgrounds,
    prepare_yolo_dataset_from_realistic
)


def train_single_yolo11_model(
    source_data_dir: str,
    model_variant: str = 'yolo11s',
    train_epochs: int = 75,
    project_name: str = "yolo11_trained_models"
):
    """
    Train a single YOLO11 model variant.
    
    Args:
        source_data_dir: Path to source data directory
        model_variant: YOLO11 variant to train (e.g., 'yolo11s', 'yolo11n', 'yolo11m')
        train_epochs: Number of training epochs
        project_name: Project directory name for saving results
    """
    
    print("🚀 Starting Single YOLO11 Defect Detection Training")
    print("=" * 60)
    
    # Display available models
    print(f"\n📋 Available YOLO11 Models:")
    available_models = YOLO11DefectDetector.list_available_models()
    for variant, info in available_models.items():
        if info['specs']:
            status = "✅ SELECTED" if variant == model_variant else "  "
            print(f"{status} {variant}: {info['specs']['size']} - {info['specs']['use_case']}")
    
    # Validate requested model
    if model_variant not in available_models:
        print(f"❌ Invalid model variant: {model_variant}")
        print(f"Available options: {list(available_models.keys())}")
        return None
    
    # Display model info
    detector = YOLO11DefectDetector(model_variant)
    info = detector.get_model_info()
    print(f"\n🎯 Training Model: {model_variant.upper()}")
    print(f"   Type: {info['type']} model")
    print(f"   Size: {info['specs'].get('size', 'Unknown')}")
    print(f"   Use Case: {info['specs'].get('use_case', 'Unknown')}")
    print(f"   Speed: {info['specs'].get('speed', 'Unknown')}")
    print(f"   Accuracy: {info['specs'].get('accuracy', 'Unknown')}")
    
    # Step 1: Organize dataset
    print("\n📁 Step 1: Organizing dataset...")
    classification_dataset = "defect_detection_dataset"
    organize_dataset(source_data_dir, classification_dataset)
    
    # Step 2: Create realistic training data
    print("\n🎨 Step 2: Creating realistic training data...")
    realistic_train_dir = "realistic_training_data"
    augment_with_backgrounds(
        source_data_dir,
        realistic_train_dir,
        objects_per_image=(1, 4),
        images_per_object=5,
        multi_object_scenes=150
    )
    
    # Step 3: Prepare YOLO dataset
    print("\n🎯 Step 3: Preparing YOLO11 dataset...")
    yolo_dataset_yaml = prepare_yolo_dataset_from_realistic(
        realistic_train_dir, "yolo11_dataset"
    )
    
    # Step 4: Train the model
    print(f"\n🤖 Step 4: Training {model_variant.upper()}...")
    
    try:
        # Train model
        best_weights = detector.train(
            dataset_yaml=yolo_dataset_yaml,
            epochs=train_epochs,
            project=project_name,
            name=f'{model_variant}_defect_detector'
        )
        
        print(f"✅ Training completed successfully!")
        print(f"💾 Best weights: {best_weights}")
        
        # Step 5: Evaluate on test data if available
        test_dir = f"{source_data_dir}/both"
        if os.path.exists(test_dir):
            print(f"\n📊 Step 5: Evaluating {model_variant} on test data...")
            eval_results = evaluate_yolo11_model(detector, test_dir)
            
            print(f"\n📈 Evaluation Results:")
            print(f"   Detection Rate: {eval_results.get('detection_rate', 0):.2%}")
            print(f"   Avg Inference Time: {eval_results.get('avg_inference_time', 0):.3f}s")
            print(f"   Total Images: {eval_results.get('total_images', 0)}")
            print(f"   Images with Detections: {eval_results.get('images_with_detections', 0)}")
            print(f"   Total Detections: {eval_results.get('total_detections', 0)}")
            
            return {
                'status': 'success',
                'model_variant': model_variant,
                'weights_path': best_weights,
                'evaluation': eval_results
            }
        else:
            print(f"\n⚠️  No test directory found at {test_dir}")
            print(f"   Skipping evaluation step")
            
            return {
                'status': 'success',
                'model_variant': model_variant,
                'weights_path': best_weights,
                'evaluation': None
            }
            
    except Exception as e:
        print(f"❌ Training failed: {e}")
        import traceback
        print(f"   Detailed error: {traceback.format_exc()}")
        return {
            'status': 'failed',
            'model_variant': model_variant,
            'error': str(e)
        }


def quick_system_check():
    """Quick system check and model selection."""
    print("🔍 System Check")
    print("=" * 30)
    
    # Check system capabilities
    capabilities = YOLO11DefectDetector.check_system_capabilities()
    
    print(f"CUDA Available: {capabilities['cuda_available']}")
    print(f"CPU Cores: {capabilities['cpu_cores']}")
    print(f"Recommended Device: {capabilities['recommended_device']}")
    print(f"Recommended Batch Size: {capabilities['recommended_batch_size']}")
    
    if capabilities['cuda_available']:
        print(f"GPU: {capabilities.get('cuda_device_name', 'Unknown')}")
        print(f"GPU Memory: {capabilities.get('cuda_memory_gb', 0):.1f} GB")
    
    # Show recommendations
    print(f"\n💡 Model Recommendations:")
    if capabilities['cuda_available']:
        print(f"   Recommended: yolo11s, yolo11m (GPU available)")
    else:
        print(f"   Recommended: yolo11n, yolo11s (CPU only)")
    
    return capabilities


def validate_data_directory(data_dir: str) -> bool:
    """Validate that the data directory exists and has the expected structure."""
    if not os.path.exists(data_dir):
        print(f"❌ Data directory not found: {data_dir}")
        return False
    
    # Check for expected subdirectories
    expected_dirs = ['defect', 'normal', 'both']
    found_dirs = []
    
    for subdir in expected_dirs:
        subdir_path = os.path.join(data_dir, subdir)
        if os.path.exists(subdir_path):
            found_dirs.append(subdir)
            file_count = len([f for f in os.listdir(subdir_path) 
                            if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
            print(f"✅ Found {subdir}/ with {file_count} images")
    
    if not found_dirs:
        print(f"❌ No expected subdirectories found in {data_dir}")
        print(f"   Expected: {expected_dirs}")
        print(f"   Available: {os.listdir(data_dir) if os.path.exists(data_dir) else 'None'}")
        return False
    
    return True


if __name__ == "__main__":
    print("🎯 YOLO11 Single Model Training Pipeline")
    print("=" * 50)
    
    # Step 0: System check
    capabilities = quick_system_check()
    
    # Configuration - UPDATE THESE PATHS
    source_dir = "data/choco-pie"  # Update this to your actual data directory
    
    # Model selection - CHANGE THIS TO TRAIN DIFFERENT MODELS
    selected_model = "yolo11s"  # Options: yolo11n, yolo11s, yolo11m, yolo11l, yolo11x
    
    # Training configuration
    training_epochs = 50  # Adjust based on your needs
    
    print(f"\n⚙️  Configuration:")
    print(f"   Data Directory: {source_dir}")
    print(f"   Selected Model: {selected_model}")
    print(f"   Training Epochs: {training_epochs}")
    
    # Validate data directory
    print(f"\n📂 Validating data directory...")
    if not validate_data_directory(source_dir):
        print(f"\n❌ Please fix the data directory path and structure.")
        print(f"   Current directory contents:")
        for item in os.listdir("."):
            if os.path.isdir(item):
                print(f"     📂 {item}/")
        exit(1)
    
    # Confirm training
    print(f"\n🚀 Ready to train {selected_model.upper()} for {training_epochs} epochs")
    print(f"   This may take some time depending on your hardware...")
    
    # Start training
    try:
        results = train_single_yolo11_model(
            source_data_dir=source_dir,
            model_variant=selected_model,
            train_epochs=training_epochs
        )
        
        # Final summary
        print(f"\n🎉 Training Pipeline Complete!")
        print(f"=" * 40)
        
        if results['status'] == 'success':
            print(f"✅ {selected_model.upper()} trained successfully")
            print(f"💾 Weights saved to: {results['weights_path']}")
            
            if results['evaluation']:
                eval_data = results['evaluation']
                print(f"📊 Performance: {eval_data.get('detection_rate', 0):.1%} detection rate")
                print(f"⚡ Speed: {eval_data.get('avg_inference_time', 0):.3f}s average inference")
            
            print(f"\n🔧 To use this model in your application:")
            print(f"   1. Update yolo_model.py model_path to: {results['weights_path']}")
            print(f"   2. Test with the detection page in your web app")
            
        else:
            print(f"❌ Training failed: {results.get('error', 'Unknown error')}")
            
    except Exception as e:
        print(f"\n❌ Pipeline failed: {e}")
        import traceback
        print(f"   Detailed error: {traceback.format_exc()}")
    
    print(f"\n" + "="*50)
    print(f"📝 To train a different model:")
    print(f"   Change 'selected_model' variable to one of:")
    print(f"   • yolo11n (fastest, smallest)")
    print(f"   • yolo11s (recommended balance)")  
    print(f"   • yolo11m (higher accuracy)")
    print(f"   • yolo11l (even higher accuracy)")
    print(f"   • yolo11x (maximum accuracy)")
    print(f"   Or use segmentation: yolo11s-seg, yolo11m-seg")

🎯 YOLO11 Single Model Training Pipeline
🔍 System Check
CUDA Available: False
CPU Cores: 16
Recommended Device: cpu
Recommended Batch Size: 4

💡 Model Recommendations:
   Recommended: yolo11n, yolo11s (CPU only)

⚙️  Configuration:
   Data Directory: data/choco-pie
   Selected Model: yolo11s
   Training Epochs: 50

📂 Validating data directory...
✅ Found defect/ with 26 images
✅ Found both/ with 11 images

🚀 Ready to train YOLO11S for 50 epochs
   This may take some time depending on your hardware...
🚀 Starting Single YOLO11 Defect Detection Training

📋 Available YOLO11 Models:
✅ SELECTED yolo11s: ~22MB - Production (Recommended)
   yolo11s-seg: ~22MB - Production (Recommended)
   yolo11s-obb: ~22MB - Production (Recommended)
   yolo11s-cls: ~22MB - Production (Recommended)

🎯 Training Model: YOLO11S
   Type: detection model
   Size: ~22MB
   Use Case: Production (Recommended)
   Speed: Fast
   Accuracy: Very Good

📁 Step 1: Organizing dataset...
Dataset organized into defect_detection_d

train: Scanning /mnt/mainhold/Downloads-Main/abb-capstone/yolo11_dataset/labels/train... 227 images, 0 backgrounds, 0 corrupt: 100%|██████████| 227/227 [00:00<00:00, 8854.50it/s]

train: New cache created: /mnt/mainhold/Downloads-Main/abb-capstone/yolo11_dataset/labels/train.cache
albumentations: Blur(p=0.01, blur_limit=(3, 7)), MedianBlur(p=0.01, blur_limit=(3, 7)), ToGray(p=0.01, method='weighted_average', num_output_channels=3), CLAHE(p=0.01, clip_limit=(1.0, 4.0), tile_grid_size=(8, 8))
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 15975.7±3730.7 MB/s, size: 167.8 KB)



/home/dealoux/.cache/pypoetry/virtualenvs/abbvisionsystem-7xechF3d-py3.12/lib/python3.12/site-packages/torch/utils/data/dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
val: Scanning /mnt/mainhold/Downloads-Main/abb-capstone/yolo11_dataset/labels/val... 48 images, 0 backgrounds, 0 corrupt: 100%|██████████| 48/48 [00:00<00:00, 9018.39it/s]

val: New cache created: /mnt/mainhold/Downloads-Main/abb-capstone/yolo11_dataset/labels/val.cache
Plotting labels to yolo11_trained_models/yolo11s_defect_detector/labels.jpg... 


...


      31/50         0G     0.1475     0.2385     0.8899          6        640: 100%|██████████| 29/29 [00:37<00:00,  1.29s/it]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:02<00:00,  1.06it/s]

                   all         48         52      0.998          1      0.995      0.995
EarlyStopping: Training stopped early as no improvement observed in last 15 epochs. Best results observed at epoch 16, best model saved as best.pt.
To update EarlyStopping(patience=15) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.



31 epochs completed in 0.353 hours.
Optimizer stripped from yolo11_trained_models/yolo11s_defect_detector/weights/last.pt, 19.2MB
Optimizer stripped from yolo11_trained_models/yolo11s_defect_detector/weights/best.pt, 19.2MB

Validating yolo11_trained_models/yolo11s_defect_detector/weights/best.pt...
Ultralytics 8.3.141 🚀 Python-3.12.11 torch-2.7.0+cu126 CPU (AMD Ryzen 7 9700X 8-Core Processor)
YOLO11s summary (fused): 100 layers, 9,413,574 parameters, 0 gradients, 21.3 GFLOPs


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:02<00:00,  1.22it/s]


                   all         48         52      0.998          1      0.995      0.995
                normal         26         27      0.997          1      0.995      0.995
                defect         25         25      0.998          1      0.995      0.995
Speed: 0.7ms preprocess, 47.3ms inference, 0.0ms loss, 0.1ms postprocess per image
Results saved to yolo11_trained_models/yolo11s_defect_detector
✅ YOLO11S training completed!
⏱️  Training time: 1276.2 seconds (21.3 minutes)
💾 Best weights saved to: yolo11_trained_models/yolo11s_defect_detector/weights/best.pt
✅ Best weights file confirmed: 18.3 MB
✅ Training completed successfully!
💾 Best weights: yolo11_trained_models/yolo11s_defect_detector/weights/best.pt

📊 Step 5: Evaluating yolo11s on test data...
🔍 Evaluating yolo11s on 11 images...

📈 Evaluation Results:
   Detection Rate: 100.00%
   Avg Inference Time: 0.068s
   Total Images: 11
   Images with Detections: 11
   Total Detections: 27

🎉 Training Pipeline Complete!
✅